In [1]:
# This cell is mandatory in all Dymos documentation notebooks.
missing_packages = []
try:
    import openmdao.api as om  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !python -m pip install openmdao[notebooks]
    else:
        missing_packages.append('openmdao')
try:
    import dymos as dm  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !python -m pip install dymos
    else:
        missing_packages.append('dymos')
try:
    import pyoptsparse  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !pip install -q condacolab
        import condacolab
        condacolab.install_miniconda()
        !conda install -c conda-forge pyoptsparse
    else:
        missing_packages.append('pyoptsparse')
if missing_packages:
    raise EnvironmentError('This notebook requires the following packages '
                           'please install them and restart this notebook\'s runtime: {",".join(missing_packages)}')

# The Transcriptions API

The transcription is the means by which Dymos converts the problem of finding a continuous control function which optimizes the operation of some system into a discrete optimization problem which can be solved by an off-the-shelf optimizer.

Dymos currently supports the following transcriptions

- Radau Pseudospectral Method
- Gauss-Lobatto Collocation
- Explicit Shooting

## Radau Options

In [2]:
import openmdao.api as om
import dymos as dm
tx = dm.transcriptions.Radau(num_segments=1, order=5)
om.show_options_table(tx)

Option,Default,Acceptable Values,Acceptable Types,Description
compressed,True,"[True, False]",['bool'],"Use compressed transcription, meaning state and control valuesat segment boundaries are not duplicated on input. This implicitly enforces value continuity between segments but in some cases may make the problem more difficult to solve."
num_segments,1,N/A,['int'],Number of segments
order,5,N/A,"['int', 'integer', 'Sequence', 'ndarray']",Order of the state transcription. The order of the control transcription is `order - 1`.
segment_ends,N/A,N/A,"['Sequence', 'ndarray']",Locations of segment ends or None for equally spaced segments
solve_segments,False,"[False, 'forward', 'backward']",N/A,"Applies 'solve_segments' behavior to _all_ states in the Phase. If 'forward', collocation defects within each segment are solved with a Newton solver by fixing the initial value in the phase (if using compressed transcription) or segment (if not using compressed transcription). This provides a forward shooting (or multiple shooting) method. If 'backward', the final value in the phase or segment is fixed and a solver finds the other ones to mimic reverse propagation. Set to False (the default) to explicitly disable the use of a solver to converge the state time history."


## GaussLobatto Options

In [3]:
tx = dm.transcriptions.GaussLobatto(num_segments=1, order=5)
om.show_options_table(tx)

Option,Default,Acceptable Values,Acceptable Types,Description
compressed,True,"[True, False]",['bool'],"Use compressed transcription, meaning state and control valuesat segment boundaries are not duplicated on input. This implicitly enforces value continuity between segments but in some cases may make the problem more difficult to solve."
num_segments,1,N/A,['int'],Number of segments
order,[5],N/A,"['int', 'integer', 'Sequence', 'ndarray']",Order of the state transcription. The order of the control transcription is `order - 1`.
segment_ends,N/A,N/A,"['Sequence', 'ndarray']",Locations of segment ends or None for equally spaced segments
solve_segments,False,"[False, 'forward', 'backward']",N/A,"Applies 'solve_segments' behavior to _all_ states in the Phase. If 'forward', collocation defects within each segment are solved with a Newton solver by fixing the initial value in the phase (if using compressed transcription) or segment (if not using compressed transcription). This provides a forward shooting (or multiple shooting) method. If 'backward', the final value in the phase or segment is fixed and a solver finds the other ones to mimic reverse propagation. Set to False (the default) to explicitly disable the use of a solver to converge the state time history."


## Birkhoff Options

In [4]:
tx = dm.transcriptions.Birkhoff(num_nodes=5, grid_type='cgl')
om.show_options_table(tx)

Option,Default,Acceptable Values,Acceptable Types,Description
compressed,True,"[True, False]",['bool'],"Use compressed transcription, meaning state and control valuesat segment boundaries are not duplicated on input. This implicitly enforces value continuity between segments but in some cases may make the problem more difficult to solve."
grid_type,cgl,"['cgl', 'lgl']",N/A,"Specifies which type of grid is to be used. Options are Chebyshev-Gauss-Lobatto (""cgl"") and Legendre-Gauss-Lobatto (""lgl"")"
num_nodes,5,N/A,"['int', 'integer']",The number of nodes in the grid
num_segments,**Required**,N/A,['int'],Number of segments
order,3,N/A,"['int', 'integer', 'Sequence', 'ndarray']",Order of the state transcription. The order of the control transcription is `order - 1`.
segment_ends,N/A,N/A,"['Sequence', 'ndarray']",Locations of segment ends or None for equally spaced segments
solve_segments,False,"[False, 'forward', 'backward']",N/A,"Applies 'solve_segments' behavior to _all_ states in the Phase. If 'forward', collocation defects within each segment are solved with a Newton solver by fixing the initial value in the phase (if using compressed transcription) or segment (if not using compressed transcription). This provides a forward shooting (or multiple shooting) method. If 'backward', the final value in the phase or segment is fixed and a solver finds the other ones to mimic reverse propagation. Set to False (the default) to explicitly disable the use of a solver to converge the state time history."


## ExplicitShooting Options

In [5]:
tx = dm.transcriptions.ExplicitShooting(num_segments=1, order=5)
om.show_options_table(tx)

Option,Default,Acceptable Values,Acceptable Types,Description,Deprecation
atol,1e-06,N/A,['float'],,N/A
compressed,True,"[True, False]",['bool'],"Use compressed transcription, meaning state and control valuesat segment boundaries are not duplicated on input. This implicitly enforces value continuity between segments but in some cases may make the problem more difficult to solve.",N/A
control_interp,vandermonde,"['vandermonde', 'barycentric', 'cubic']",N/A,"Control interpolation algorithm, one of either ""vandermonde"", ""barycentric"", or ""cubic"". In general, Vandermonde is faster but Barycentric is necessary for the Birkhoff transcription where the number of nodes per segment can exceed 20 to 30. Cubic uses the scipy interpolate cubic spline method and is typically the fastest of the three",N/A
first_step,N/A,N/A,['float'],,N/A
grid,"GaussLobattoGrid(num_seg=1, order=[5]) at <140203048708176>",N/A,"['GaussLobattoGrid', 'ChebyshevGaussLobattoGrid', 'RadauGrid', 'BirkhoffGrid', 'str']",The grid distribution used to layout the control inputs and provide the default output nodes.,N/A
max_step,inf,N/A,['float'],,N/A
method,DOP853,N/A,['str'],The integration method used.,N/A
num_segments,1,N/A,['int'],Number of segments,N/A
num_steps_per_segment,N/A,N/A,['int'],Number of integration steps in each segment,Option `num_steps_per_segment is deprecated. ExplicitShooting now uses adaptive-step methods.
order,5,N/A,"['int', 'Sequence', 'ndarray']",Order of the state transcription. The order of the control transcription is `order - 1`.,N/A


## PicardShooting Options

In [6]:
tx = dm.transcriptions.PicardShooting(num_segments=1, nodes_per_seg=5)
om.show_options_table(tx)

Option,Default,Acceptable Values,Acceptable Types,Description
compressed,True,"[True, False]",['bool'],"Use compressed transcription, meaning state and control valuesat segment boundaries are not duplicated on input. This implicitly enforces value continuity between segments but in some cases may make the problem more difficult to solve."
grid_type,cgl,"['cgl', 'lgl']",N/A,"Specifies which type of grid is to be used. Options are Chebyshev-Gauss-Lobatto (""cgl"") and Legendre-Gauss-Lobatto (""lgl"")"
ms_linear_solver,LN: Direct,N/A,N/A,Linear solver used to linearize the Picard iteration subsystem between segments.
ms_nonlinear_solver,NL: NLBGS,N/A,N/A,Nonlinear solver used to resolve Picard iteration differences between segments.
nodes_per_seg,5,N/A,['int'],The number of segments in the grid.
num_segments,1,N/A,['int'],The number of segments in the grid.
ode_linear_solver,LN: Direct,N/A,N/A,Linear solver used to linearize the Picard iteration subsystem on each segment.
ode_nonlinear_solver,NL: NLBGS,N/A,N/A,Nonlinear solver used to resolve Picard iteration on each segment.
order,3,N/A,"['int', 'integer', 'Sequence', 'ndarray']",Order of the state transcription. The order of the control transcription is `order - 1`.
segment_ends,N/A,N/A,"['Sequence', 'ndarray']",Locations of segment ends or None for equally spaced segments
